# CMIP6 metadata and clisops regridding

Run bilinear regridding through `clisops.ops.regrid.regrid` with the xESMF/ESMPy backend. Each synthetic source is written to NetCDF and decoded with xarray. All three sources use the same smaller target grid inside their geographic coverage.

In [1]:
from importlib.metadata import version

import esmpy
import ipytest
import numpy as np
import pytest
import xarray as xr
import xesmf
from clisops import CONFIG
from clisops.ops.regrid import regrid

ipytest.autoconfig(raise_on_error=True, magics=False, addopts=())

for package in ("clisops", "xarray", "xesmf", "esmpy"):
    print(f"{package} {version(package)}")

clisops 0.18.1
xarray 2025.10.1
xesmf 0.9.2
esmpy 8.9.1


## 3.7 Horizontal x/y coordinate units

The finding flags `degrees` on axes with the standard names `projection_x_coordinate` and `projection_y_coordinate`. The synthetic variant changes only the two unit attributes from `m` to `degrees`. The metre-valued curvilinear control and the variant share the same 2-D geographic latitude and longitude, native axis values, and data. A regular latitude/longitude grid provides a separate control for the operator and backend.

In [2]:
LAT = {"standard_name": "latitude", "units": "degrees_north"}
LON = {"standard_name": "longitude", "units": "degrees_east"}
time = np.array(["2000-01-01", "2000-01-02"], dtype="datetime64[ns]")
lat = np.array([44.0, 45.0, 46.0, 47.0])
lon = np.array([-12.0, -11.0, -10.0, -9.0])
lon_grid, lat_grid = np.meshgrid(lon, lat)
lat_curved = lat_grid + 0.08 * (lon_grid - lon[0])
lon_curved = lon_grid + 0.06 * (lat_grid - lat[0])


def temperatures(latitude, longitude):
    return 260.0 + latitude + 2.0 * longitude + np.array([0.0, 10.0])[:, None, None]


regular = xr.Dataset(
    {"tas": (("time", "lat", "lon"), temperatures(lat_grid, lon_grid))},
    coords={"time": time, "lat": ("lat", lat, LAT), "lon": ("lon", lon, LON)},
)
curvilinear = xr.Dataset(
    {"tas": (("time", "y", "x"), temperatures(lat_curved, lon_curved))},
    coords={
        "time": time,
        "x": ("x", np.arange(4) * 80_000,
              {"standard_name": "projection_x_coordinate", "units": "m"}),
        "y": ("y", np.arange(4) * 110_000,
              {"standard_name": "projection_y_coordinate", "units": "m"}),
        "lat": (("y", "x"), lat_curved, LAT),
        "lon": (("y", "x"), lon_curved, LON),
    },
)
incorrect_units = curvilinear.copy(deep=True)
for axis in ("x", "y"):
    incorrect_units[axis].attrs["units"] = "degrees"

target = xr.Dataset(coords={
    "lat": ("lat", [44.6, 46.2], LAT),
    "lon": ("lon", [-11.4, -10.4, -9.4], LON),
})

## Pytest tests

Each run gets an empty weights cache so clisops calculates weights independently for the two curvilinear inputs. Each test checks the target grid, the two input times, and finite values. The metadata comparison depends on the valid curvilinear control and uses an absolute tolerance of `1e-10`.

The backend imports and the regular-grid test help diagnose environment or operator failures. If the curvilinear control fails, the units comparison is inconclusive. A difference with the curvilinear control passing isolates the units change in this fixture.

In [3]:
ipytest.clean()


def remap(source, directory):
    path = directory / "source.nc"
    source.to_netcdf(path, engine="netcdf4")
    with xr.open_dataset(path, engine="netcdf4") as dataset:
        decoded = dataset.load()

    for axis in ("x", "y"):
        if axis in source.coords:
            assert decoded[axis].attrs["units"] == source[axis].attrs["units"]

    weights = directory / "weights"
    weights.mkdir()
    with pytest.MonkeyPatch.context() as patch:
        patch.setitem(CONFIG["clisops:grid_weights"], "local_weights_dir", str(weights))
        outputs = regrid(decoded, grid=target.copy(deep=True), method="bilinear", output_type="xarray")
    assert len(outputs) == 1
    return outputs[0]


def assert_target(result):
    assert result.tas.dims == ("time", "lat", "lon")
    assert result.sizes == {"time": 2, "lat": 2, "lon": 3}
    np.testing.assert_array_equal(result.time, time)
    np.testing.assert_array_equal(result.lat, target.lat)
    np.testing.assert_array_equal(result.lon, target.lon)
    assert np.isfinite(result.tas.values).all()


@pytest.fixture(scope="module")
def curvilinear_control(tmp_path_factory):
    return remap(curvilinear, tmp_path_factory.mktemp("curvilinear-control"))


def test_regular_grid(tmp_path):
    assert_target(remap(regular, tmp_path))


def test_curvilinear_grid(curvilinear_control):
    assert_target(curvilinear_control)


def test_projected_axis_units(curvilinear_control, tmp_path):
    actual = remap(incorrect_units, tmp_path)
    assert_target(actual)
    np.testing.assert_allclose(actual.tas, curvilinear_control.tas, rtol=0, atol=1e-10)
    difference = np.abs(actual.tas.values - curvilinear_control.tas.values).max()
    print(f"Maximum absolute difference from the curvilinear control: {difference:.3g}")

## Run pytest

In [4]:
ipytest.run("-v", "-rP", "--color=no", "-p", "no:cacheprovider")

======================================= test session starts ========================================
platform darwin -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- /private/tmp/c3s-cmip6-update-tests-01a1cbf2/bin/python
rootdir: /Users/pingu/Documents/GitHub/macpingu/c3s-cmip6-update-tests/notebooks
plugins: platformdirs-4.14.5, anyio-4.16.0
collecting ... collected 3 items

t_4896b48d76834b6b8c6e84c51bd9c1e9.py::test_regular_grid PASSED                              [ 33%]
t_4896b48d76834b6b8c6e84c51bd9c1e9.py::test_curvilinear_grid PASSED                          [ 66%]
t_4896b48d76834b6b8c6e84c51bd9c1e9.py::test_projected_axis_units PASSED                      [100%]

========================================= warnings summary =========================================
t_4896b48d76834b6b8c6e84c51bd9c1e9.py: 12 warnings
  /private/tmp/c3s-cmip6-update-tests-01a1cbf2/lib/python3.13/site-packages/clisops/utils/dataset_utils.py:1853: UserWarning: For coordinate variable 'lat' no bounds c

<ExitCode.OK: 0>

## Results and limitations

All three tests passed. Each output kept both times and matched the 2 × 3 target grid. The degree-units variant and the metre-valued curvilinear control had a maximum absolute difference of 0. Clisops emitted 24 warnings about absent cell bounds, 12 for latitude and 12 for longitude.

This run covers bilinear regridding on small single-file inputs with valid geographic coordinates. Other methods, grids, backend versions, and real datasets may behave differently.